In [26]:
import xarray as xr
import pandas as pd
import dask
import glob
import os

In [2]:
from dask.distributed import Client

from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=32, threads_per_worker=1)
client = Client(cluster)
client

/glade/work/qingyuany/conda-envs/ml_env/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44385 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/qingyuany/Work/proxy/44385/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/qingyuany/Work/proxy/44385/status,Workers: 32
Total threads: 32,Total memory: 64.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:35803,Workers: 32
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/qingyuany/Work/proxy/44385/status,Total threads: 32
Started: Just now,Total memory: 64.00 GiB
Comm: tcp://127.0.0.1:42139,Total threads: 1
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/qingyuany/Work/proxy/37211/status,Memory: 2.00 GiB
Nanny: tcp://127.0.0.1:40119,


In [3]:
voi = ['FSNTOA', 'FLUT', 'FSNT', 'FLNT', 'SWCF', 'LWCF', 'PRECT', 'TGCLDLWP', 'FSNTC', 'FLUTC', 'TMQ', 'LHFLX']


In [14]:
archive_paths = "/glade/derecho/scratch/qingyuany/archive"
cases = glob.glob(os.path.join(archive_paths, "v0_seg2*"))

In [15]:
cases

['/glade/derecho/scratch/qingyuany/archive/v0_seg2.024',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.029',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.049',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.017',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.002',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.050',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.012',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.006',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.047',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.025',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.018',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.003',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.040',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.046',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.038',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.033',
 '/glade/derecho/scratch/qingyuany/archive/v0_seg2.001',
 '/glade/derecho/scratch/qingyu

In [16]:
ds_list = []
run_id = []

for case in cases:
    
    run_id.append(int(case.split("/")[-1][-3:]))
    print(f'{case} : {int(case.split("/")[-1][-3:])}')
    time_files = glob.glob(os.path.join(case, "atm/hist",  "*h0a*"))
    sim_per = xr.open_mfdataset(time_files, combine='by_coords', chunks = {"time":1})
    
    sim_par = sim_per[voi].mean(dim = "time")
    
    ds_list.append(sim_par)

/glade/derecho/scratch/qingyuany/archive/v0_seg2.024 : 24
/glade/derecho/scratch/qingyuany/archive/v0_seg2.029 : 29
/glade/derecho/scratch/qingyuany/archive/v0_seg2.049 : 49
/glade/derecho/scratch/qingyuany/archive/v0_seg2.017 : 17
/glade/derecho/scratch/qingyuany/archive/v0_seg2.002 : 2
/glade/derecho/scratch/qingyuany/archive/v0_seg2.050 : 50
/glade/derecho/scratch/qingyuany/archive/v0_seg2.012 : 12
/glade/derecho/scratch/qingyuany/archive/v0_seg2.006 : 6
/glade/derecho/scratch/qingyuany/archive/v0_seg2.047 : 47
/glade/derecho/scratch/qingyuany/archive/v0_seg2.025 : 25
/glade/derecho/scratch/qingyuany/archive/v0_seg2.018 : 18
/glade/derecho/scratch/qingyuany/archive/v0_seg2.003 : 3
/glade/derecho/scratch/qingyuany/archive/v0_seg2.040 : 40
/glade/derecho/scratch/qingyuany/archive/v0_seg2.046 : 46
/glade/derecho/scratch/qingyuany/archive/v0_seg2.038 : 38
/glade/derecho/scratch/qingyuany/archive/v0_seg2.033 : 33
/glade/derecho/scratch/qingyuany/archive/v0_seg2.001 : 1
/glade/derecho/scr

In [17]:
ds_ta = xr.concat(ds_list, dim=pd.Index(run_id, name="ppe_ind"))
ds_ta = ds_ta.sortby("ppe_ind")

In [18]:
ds_ta = ds_ta.compute()

In [19]:
ds_ta["RESTOM"] = ds_ta["FSNT"] - ds_ta["FLNT"]
ds_ta["RESTOA"] = ds_ta["FSNTOA"] - ds_ta["FLUT"]


In [21]:
#ds_ta.to_netcdf("/glade/work/qingyuany/cam7_re/v0/seg2_ppe_out.nc")

In [29]:
dsseg1 = xr.open_dataset('/glade/work/qingyuany/cam7_re/v0/seg1_ppe_out.nc')
dsseg2 = xr.open_dataset('/glade/work/qingyuany/cam7_re/v0/seg2_ppe_out.nc')

dsseg2["ppe_ind"] = dsseg2["ppe_ind"] + 50


FileNotFoundError: [Errno 2] No such file or directory: '/glade/work/qingyuany/cam7_re/v0/seg1_ppe_out.nc'

In [ ]:
combined = xr.concat([dsseg1, dsseg2], dim = "ppe_ind")

In [32]:
combined.ppe_ind

<xarray.DataArray 'ppe_ind' (ppe_ind: 98)> Size: 784B
array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,
        15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,
        29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,  42,
        43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,  56,
        57,  58,  59,  62,  63,  64,  65,  66,  67,  68,  69,  70,  71,  72,
        73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,
        87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100])
Coordinates:
  * ppe_ind  (ppe_ind) int64 784B 1 2 3 4 5 6 7 8 9 ... 93 94 95 96 97 98 99 100

In [35]:
#combined.to_netcdf("/glade/work/qingyuany/cam7_re/v0/ppe_out.nc")
combined = xr.open_dataset("/glade/work/qingyuany/cam7_re/v0/ppe_out.nc")


In [36]:
import pandas as pd
paraseg1 = pd.read_csv('/glade/work/qingyuany/cam7_re/v0/sel_para_seg1.csv', index_col = 0)
paraseg2 = pd.read_csv('/glade/work/qingyuany/cam7_re/v0/sel_para_seg2.csv', index_col = 0)


In [39]:
paraseg1.index = paraseg1.index + 1
paraseg2.index = paraseg2.index + 1


In [40]:
para_combined = pd.concat([paraseg1, paraseg2], axis = 0)
para_combined.shape

(100, 38)

In [52]:
para_combined.loc[combined.ppe_ind].to_csv("/glade/work/qingyuany/cam7_re/v0/para_consist.csv")